In [3]:
import os 

print(len(os.listdir('/home/psh/af3_output_processed')))
print(len(os.listdir('/home/psh/benchmark_after210930/pdb')))
print(set([file + '.pdb' for file in os.listdir('/home/psh/af3_output_processed')]) - set(os.listdir('/home/psh/benchmark_after210930/pdb')))

61
59
{'8dke_A_B_P.pdb', '8vzo_B_D_A.pdb'}


## af3 output에서 heavy와 light chain만 뽑기

In [11]:
import os 

def reorder_chains_in_pdb(input_pdb_file, output_pdb_file, hchain, lchain, agchains):
    """
    PDB 파일의 체인을 Heavy Chain, Light Chain, Antigen Chain 순으로 재배치합니다.
    """
    with open(input_pdb_file, 'r') as infile:
        lines = infile.readlines()

    # 체인별 데이터를 저장
    chains = {hchain: [], lchain: []}
    for agchain in agchains:
        chains[agchain] = []

    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            chain_id = line[21]  # 22번째 문자로 체인 ID 추출
            if chain_id in chains:
                chains[chain_id].append(line)

    # 체인의 순서를 HCHAIN, LCHAIN, AGCHAIN 순으로 정렬
    if lchain == '#':
        reordered_lines = chains[hchain]
    else:
        reordered_lines = chains[hchain] + chains[lchain]

    for agchain in agchains:
        reordered_lines += chains[agchain]

    # 재배열된 파일 저장
    with open(output_pdb_file, 'w') as outfile:
        outfile.writelines(reordered_lines)

    print(f"Reordered PDB saved to {output_pdb_file}")


target_root_dir = '/home/psh/af3_output'
output_root_dir = '/home/psh/af3_output_processed'
os.makedirs(output_root_dir, exist_ok=True)

for file in os.listdir(target_root_dir):
    file_dir = os.path.join(target_root_dir, file)
    output_file_dir = os.path.join(output_root_dir, file)
    os.makedirs(output_file_dir, exist_ok=True)
    for i, sample in enumerate(os.listdir(file_dir)):
        target_path = os.path.join(file_dir, sample, 'sample_1.pdb')
        os.makedirs(os.path.join(output_file_dir, sample), exist_ok=True)
        output_path = os.path.join(output_file_dir, sample, 'sample_1.pdb')

        chains = file.split('_')[1:]
        chains[-1] = chains[-1].replace('pdb', '')
        hchain = chains[0]
        lchain = chains[1]
        agchains = [agchain for agchain in chains[2]]

        reorder_chains_in_pdb(target_path, output_path, hchain, lchain, agchains)


Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_0/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_1/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_2/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_3/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_4/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_5/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_6/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_7/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_8/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_9/sample_1.pdb
Reordered PDB saved to /home/psh/af3_output_processed/7df1_F_J_C/sample_10/sample_1.pdb
Reordered PDB saved to /home/psh/af3_outpu

# gap 제거 

In [14]:
from Bio import PDB
import os

# 주어진 시퀀스 (gap 포함)
file_dir = '/home/psh/af3_output_processed_copy/8pg0_H_L_A'
pdb_file2 = '/home/psh/benchmark_after210930/pdb_only_ab/8pg0_H_L_A.pdb'
def remove_extra_residues_based_on_number_and_chain(pdb_file1, pdb_file2, output_pdb):
    # PDB 파일을 읽어들이기
    with open(pdb_file1, 'r') as f1, open(pdb_file2, 'r') as f2:
        lines1 = f1.readlines()
        lines2 = f2.readlines()

    # 두 번째 파일에서의 residue 번호와 chain ID를 세트로 저장
    residue_chain_pairs2 = set()
    for line in lines2:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            residue_number = int(line[22:26].strip())  # residue 번호는 23번째부터 26번째까지
            chain_id = line[21:22]  # chain ID는 22번째부터 23번째까지
            residue_chain_pairs2.add((residue_number, chain_id))

    # 첫 번째 파일에서 두 번째 파일에 존재하는 residue 번호와 chain ID에 해당하는 것만 필터링
    filtered_lines = []
    for line in lines1:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            residue_number = int(line[22:26].strip())
            chain_id = line[21:22]
            if (residue_number, chain_id) in residue_chain_pairs2:
                filtered_lines.append(line)

    # 수정된 내용 파일에 저장
    with open(output_pdb, 'w') as output_file:
        output_file.writelines(filtered_lines)


# 사용 예시
for sample in os.listdir(file_dir):
    input_pdb = os.path.join(file_dir, sample, 'only_ab.pdb')
    output_pdb = os.path.join(file_dir, sample, 'only_ab.pdb')
    remove_extra_residues_based_on_number_and_chain(input_pdb, pdb_file2, output_pdb)

# rechain alphafold output

In [ ]:
import argparse
import sys
import os

import warnings
warnings.filterwarnings("ignore")

# 현재 경로 및 부모 디렉터리 경로 설정
sys.path.append('/home/psh/protein-frame-flow')

# from local_igfold.test.test import *
from data.ab_metrics import *
from Bio import PDB


target_root_dir = '/home/psh/af3_output'

os.makedirs(target_root_dir, exist_ok=True)

for file in os.listdir(target_root_dir):
    file_dir = os.path.join(target_root_dir, file)

    for i, sample in enumerate(os.listdir(file_dir)):
        target_path = os.path.join(file_dir, 'sample_1.pdb')

        a = renumber_pdb(target_path)
        